CNN

If CNN only looks at a 3×3 region at a time, how does it understand the whole image?

Answer:

Because multiple convolution layers gradually combine local features into larger and more complex patterns. Early layers detect edges and corners, while deeper layers combine them into shapes, digits, or objects.

# Dataset and DataLoader

In [23]:
from torchvision import datasets # for importing dataset
from torchvision import transforms #  for making transformation in images , raw images = > tensors
from torch.utils.data import DataLoader  # making batches from datasets for loading data

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In RNN, we used squeeze(1) because the RNN expects input of shape (batch, sequence_length, input_size). The channel dimension is not required, so we remove it.

In CNN, we do not use squeeze(1) because Conv2D expects the input shape (batch, channels, height, width). The channel dimension tells the CNN how many channels the image has (1 for grayscale, 3 for RGB), so it must be preserved.

# Defining CNN model

In [24]:
import torch
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.relu = nn.ReLU()

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.fc1 = nn.Linear(64 * 7 * 7, 128)

        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(self.relu(self.conv1(x)))

        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))

        x = self.fc2(x)

        return x

Many interviewers ask:

Why do we use padding=1 with a 3×3 kernel?

Answer:

To preserve the spatial dimensions of the feature map. Without padding, each convolution would reduce the height and width of the image, causing loss of information near the borders.

=====


Because pooling reduces spatial dimensions.

It doesn't care about channels.

It simply says:

"From every 2×2 region, keep only the most important value."


-
Why do we do this?

Imagine a 1000×1000 image.

Processing every pixel in every layer would be expensive.

Pooling:

✅ Reduces computation
✅ Reduces memory
✅ Keeps the strongest features
✅ Makes the model more robust to small shifts in the image

Think of it like image compression while preserving the important information.
=======

convo , linear
All of these are linear operations.

A stack of linear operations is still equivalent to one linear operation.

That means the network cannot learn complex patterns.

ReLU introduces non-linearity, allowing the network to learn complex features like:

Curves
Digits
Faces
Objects

Without ReLU, a deep CNN would have very limited expressive power.

====
The fully connected (Linear) layer cannot accept a 4D tensor. It expects:

(batch_size, features)

So we convert each sample from:

64 feature maps

Each feature map = 7 × 7

into one long vector.

Number of features:

64 × 7 × 7 = 3136

Notice that only one image is flattened at a time. The batch size stays the same.
=====



Why -1?
x.view(x.size(0), -1)

Here,

x.size(0)

means:

Batch Size = 64

The -1 tells PyTorch:

"You calculate the remaining dimension automatically."

-
Without -1:

x = x.view(64, 3136)

❌ Code may break.

With

x = x.view(x.size(0), -1)

✅ PyTorch recalculates the correct feature size automatically.


Input Image
(64,1,28,28)

        │
        ▼
Conv2D
Extract local features
(Edges, Curves, Corners)

(64,32,28,28)

        │
        ▼
ReLU
Negative → 0
Positive → Keep

(64,32,28,28)

        │
        ▼
MaxPool
Keep strongest features
Reduce size

(64,32,14,14)

        │
        ▼
Conv2D
Learn more complex features

(64,64,14,14)

        │
        ▼
ReLU

(64,64,14,14)

        │
        ▼
MaxPool

(64,64,7,7)

        │
        ▼
Flatten

(64,3136)

        │
        ▼
FC1

(64,128)

        │
        ▼
FC2

(64,10)

        │
        ▼
CrossEntropyLoss

In [25]:
# device selection

device = "cuda" if torch.cuda.is_available else "cpu"
print(device)

cuda


In [26]:
#model initialization
model = CNNModel().to(device)

In [27]:
# loss function and optimizer

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Training

In [28]:
num_epochs = 10

for epoch in range(num_epochs):

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass
        outputs = model(images)

        # Calculate Loss
        loss = criterion(outputs, labels)

        # Clear Previous Gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update Weights
        optimizer.step()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")



Epoch [1/10], Loss: 0.0987
Epoch [2/10], Loss: 0.0149
Epoch [3/10], Loss: 0.0010
Epoch [4/10], Loss: 0.0160
Epoch [5/10], Loss: 0.0066
Epoch [6/10], Loss: 0.0128
Epoch [7/10], Loss: 0.0009
Epoch [8/10], Loss: 0.0013
Epoch [9/10], Loss: 0.0584
Epoch [10/10], Loss: 0.0012


# Testing

In [29]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)  # torch.max(..., 1) returns: Maximum value = 5.2 ,Index = 2

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test accuracy : {accuracy:.2f}%")


Test accuracy : 98.98%


# save the model

In [32]:
# from google.colab import drive
# drive.mount('/content/drive')


torch.save(model, "/content/drive/MyDrive/MNIST - DL/models/mnist_cnn.pth")

Load the model

In [31]:
# model = CNNModel()

# model.load_state_dict(torch.load("/content/drive/MyDrive/MNIST - DL/mnist_cnn.pth"))
# model.eval()

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.CNNModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.CNNModel])` or the `torch.serialization.safe_globals([__main__.CNNModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default